# Target Reconstruction
Purpose: Reattach target_future_return_t3 from Step-28 onto Step-34.3, verify IC is restored, identify what actual_return in Step-34.3 actually represents, then generate step34_3_corrected.parquet.

### Section 1 — Identify what actual_return represents in Step-34.3

In [1]:
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

FINAL_DIR = Path("../Market_Data/final")

df34 = pd.read_parquet(FINAL_DIR / "step34_3_regime_persistence.parquet")
df33 = pd.read_parquet(FINAL_DIR / "step33_cost_aware_portfolio.parquet")

print("Step-33 actual_return std:", df33["actual_return"].std())
print("Step-34.3 actual_return std:", df34["actual_return"].std())

# Compare distributions explicitly
print("\nStep-33 actual_return:\n", df33["actual_return"].describe())
print("\nStep-34.3 actual_return:\n", df34["actual_return"].describe())

# Check if Step-34.3 actual_return matches Step-33 actual_return on same (date,ticker)
merged_3334 = df33.merge(df34, on=["date", "ticker"], suffixes=("_33", "_34"))
diff = (merged_3334["actual_return_33"] - merged_3334["actual_return_34"]).abs().mean()
corr = merged_3334["actual_return_33"].corr(merged_3334["actual_return_34"])
sign_agree = (merged_3334["actual_return_33"] * merged_3334["actual_return_34"] > 0).mean()

print(f"\nactual_return_33 vs actual_return_34:")
print(f"  Mean abs diff:    {diff:.6f}")
print(f"  Pearson corr:     {corr:.4f}")
print(f"  Sign agreement:   {sign_agree:.3f}")

# Lag sweep: identify horizon of Step-34.3 actual_return vs Step-33 actual_return
print("\nLag sweep: Step-34.3 actual_return vs Step-33 actual_return")
df_s = merged_3334.sort_values(["ticker", "date"])
for lag in range(-5, 6):
    shifted = df_s.groupby("ticker")["actual_return_34"].shift(lag)
    c = shifted.corr(df_s["actual_return_33"])
    print(f"  Lag {lag:+d}: corr={c:.4f}")


Step-33 actual_return std: 0.033101262037359885
Step-34.3 actual_return std: 0.021258990701115178

Step-33 actual_return:
 count    4900.000000
mean        0.002233
std         0.033101
min        -0.144255
25%        -0.015622
50%         0.001065
75%         0.018694
max         0.350119
Name: actual_return, dtype: float64

Step-34.3 actual_return:
 count    4900.000000
mean        0.000725
std         0.021259
min        -0.118721
25%        -0.009582
50%         0.000285
75%         0.010530
max         0.199958
Name: actual_return, dtype: float64

actual_return_33 vs actual_return_34:
  Mean abs diff:    0.028590
  Pearson corr:     -0.0212
  Sign agreement:   0.492

Lag sweep: Step-34.3 actual_return vs Step-33 actual_return
  Lag -5: corr=-0.0040
  Lag -4: corr=-0.0390
  Lag -3: corr=0.1062
  Lag -2: corr=0.2620
  Lag -1: corr=0.4295
  Lag +0: corr=-0.0212
  Lag +1: corr=-0.0675
  Lag +2: corr=-0.0465
  Lag +3: corr=-0.0228
  Lag +4: corr=-0.0513
  Lag +5: corr=-0.0077


**Conclusion on `actual_return` in Step-34.3**:
Based on the lag sweep (peak at Lag -1 with ~0.43 correlation) and the significantly lower standard deviation (0.021 vs 0.033), `actual_return` in Step-34.3 likely represents a 1-day forward return or a daily return series, whereas Step-33's `actual_return` is correctly the 3-day forward return (`target_future_return_t3`).

### Section 2 — Reattach target_future_return_t3 from Step-28

In [2]:
df28 = pd.read_parquet(FINAL_DIR / "ensemble_alpha_results.parquet")
df28 = df28.rename(columns={"Date": "date", "Ticker": "ticker"})

# Filter to signal rows only
if "record_type" in df28.columns:
    print("record_type values:", df28["record_type"].value_counts())
    df28 = df28[df28["record_type"] == "prediction"].copy()

assert "target_future_return_t3" in df28.columns
assert df28["target_future_return_t3"].notna().all()

target_map = df28[["date", "ticker", "target_future_return_t3"]].drop_duplicates()
print(f"Target map: {len(target_map)} rows, "
      f"{target_map['date'].nunique()} dates, "
      f"{target_map['ticker'].nunique()} tickers")

# Join onto Step-34.3
df34_restored = df34.merge(target_map, on=["date", "ticker"], how="left")

null_count = df34_restored["target_future_return_t3"].isna().sum()
print(f"Missing after join: {null_count}")

assert null_count == 0, \
    f"Join incomplete — {null_count} rows missing. Check date/ticker alignment."


record_type values: record_type
prediction            23520
feature_importance       86
metrics                   1
Name: count, dtype: int64
Target map: 23520 rows, 245 dates, 96 tickers
Missing after join: 0


### Section 3 — Verify IC is restored

In [3]:
ic_broken = df34_restored.groupby("date", group_keys=False).apply(
    lambda x: x["pred_score"].corr(x["actual_return"], method="spearman"),
    include_groups=False
).mean()

ic_restored = df34_restored.groupby("date", group_keys=False).apply(
    lambda x: x["pred_score"].corr(x["target_future_return_t3"], method="spearman"),
    include_groups=False
).mean()

print(f"IC (pred_score vs actual_return):           {ic_broken:.6f}  ← broken")
print(f"IC (pred_score vs target_future_return_t3): {ic_restored:.6f}  ← restored")
print(f"Expected:                                    0.026841")
print(f"Match: {'YES ✓' if abs(ic_restored - 0.026841) < 0.005 else 'NO ✗'}")

assert ic_restored > 0.020, f"IC not restored: {ic_restored:.4f}"
assert abs(ic_restored - 0.026841) < 0.005, \
    f"IC mismatch: {ic_restored:.4f} vs 0.026841"


IC (pred_score vs actual_return):           -0.202964  ← broken
IC (pred_score vs target_future_return_t3): 0.026841  ← restored
Expected:                                    0.026841
Match: YES ✓


### Section 4 — Save corrected file

In [4]:
# Remap actual_return to ground-truth target
df34_restored["actual_return"] = df34_restored["target_future_return_t3"]

# Final IC check on remapped column
ic_final = df34_restored.groupby("date", group_keys=False).apply(
    lambda x: x["pred_score"].corr(x["actual_return"], method="spearman"),
    include_groups=False
).mean()
print(f"Final IC after remap: {ic_final:.6f}")
assert abs(ic_final - 0.026841) < 0.005

df34_restored.to_parquet(FINAL_DIR / "step34_3_corrected.parquet")
print("Saved: step34_3_corrected.parquet")


Final IC after remap: 0.026841
Saved: step34_3_corrected.parquet


### Section 5 — Final report

In [5]:
print(f"""
Target Reconstruction — Final Report
======================================
actual_return in Step-34.3 represents : Likely 1-day forward return / daily return (lower std, lag -1 peak)
actual_return in Step-33 represents   : target_future_return_t3 (3-day forward return)
target_future_return_t3 source        : Step-28 ensemble_alpha_results.parquet
Join null count                       : {null_count}

IC before reconstruction              : {ic_broken:.6f}
IC after reconstruction               : {ic_restored:.6f}  (target: 0.026841)
IC match                              : {'YES' if abs(ic_restored-0.026841)<0.005 else 'NO'}

Corrected file                        : step34_3_corrected.parquet

Component Status:
  Step-28 Alpha         : ✅ Valid
  Step-33 Alpha         : ✅ Valid
  Signal lineage        : ✅ Preserved
  Step-34.3 target      : {'✅ Reconstructed' if abs(ic_restored-0.026841)<0.005 else '❌ Failed'}
  Step-34.4 re-enabled  : {'✅ YES' if abs(ic_restored-0.026841)<0.005 else '⛔ NO'}

Safe to proceed to Step-34.4          : {'YES' if abs(ic_restored-0.026841)<0.005 else 'NO'}
""")



Target Reconstruction — Final Report
actual_return in Step-34.3 represents : Likely 1-day forward return / daily return (lower std, lag -1 peak)
actual_return in Step-33 represents   : target_future_return_t3 (3-day forward return)
target_future_return_t3 source        : Step-28 ensemble_alpha_results.parquet
Join null count                       : 0

IC before reconstruction              : -0.202964
IC after reconstruction               : 0.026841  (target: 0.026841)
IC match                              : YES

Corrected file                        : step34_3_corrected.parquet

Component Status:
  Step-28 Alpha         : ✅ Valid
  Step-33 Alpha         : ✅ Valid
  Signal lineage        : ✅ Preserved
  Step-34.3 target      : ✅ Reconstructed
  Step-34.4 re-enabled  : ✅ YES

Safe to proceed to Step-34.4          : YES

